# receipt-vlm on a rented GPU (Colab / Kaggle / cloud)

All logic lives in `src/receipt_vlm/`; this notebook only runs the same `make` targets as on any
machine. Target: one 16 GB GPU (T4/L4) with `configs/*/gpu16.yaml`, or ≥ 24 GB with
`gpu24plus`. Order matters: **validation for every decision, test once at the end**.

Secrets: set `HF_TOKEN` (needed only to push models) and optionally `WANDB_API_KEY` in the Colab
secrets panel.

In [ ]:
!nvidia-smi
import os

REPO = "https://github.com/StepanovArt/expense-tracker"
BRANCH = "claude/modest-carson-oyl8jv"  # or main once merged
!git clone -b {BRANCH} {REPO} /content/expense-tracker
%cd /content/expense-tracker/receipt-vlm

In [ ]:
# secrets (optional)
import contextlib

try:
    from google.colab import userdata

    for k in ("HF_TOKEN", "WANDB_API_KEY"):
        with contextlib.suppress(Exception):
            os.environ[k] = userdata.get(k)
except ImportError:
    pass
!pip -q install uv
!make setup-gpu

## 1. Data (Phase 1–2)
Adds CORD v2 (HF Hub) and WildReceipt to the frozen SROIE + synthetic splits; existing test entries cannot move (the build fails if they would).

In [ ]:
!make data
!make synth
!uv run pytest -m data -q
!cat reports/data_stats.md

## 2. Zero-shot baselines on validation (Phase 3)

In [ ]:
!make eval-baselines SPLIT=val
!make report

## 3. SFT (Phase 4)
Checkpoint selection by generation-based val F1 on a fixed 200-image subset. Try `lora.train_projector=true` as a second run and keep whichever wins on val.

In [ ]:
!make sft CONFIG=configs/sft/gpu16.yaml
# optional comparison run:
# !uv run python scripts/train_sft.py --config configs/sft/gpu16.yaml output_dir=outputs/sft_proj lora.train_projector=true
!make eval MODEL=M1 SPLIT=val EVALCFG=configs/eval/final.yaml
!make report

## 4. GRPO (Phase 6) — validation first

In [ ]:
!make grpo CONFIG=configs/grpo/gpu16.yaml
!make eval MODEL=M2 SPLIT=val EVALCFG=configs/eval/final.yaml
!make report

## 5. Frozen test set — run once (Phase 5/6)

In [ ]:
!uv run python scripts/run_eval.py --config configs/eval/final.yaml --split test --models B0 B1 B2 B3 M1 M2
!make report
!uv run python scripts/error_analysis.py --model M2 --split test

## 6. Save results back to the repo
Predictions stay in `outputs/` (gitignored, large); metrics, report and figures are committed so every number is reproducible from a command.

In [ ]:
!git add reports data/splits && git -c user.name=colab -c user.email=colab@localhost commit -m "results: GPU run" || true
# !git push   # needs a token with push access
!tar czf /content/receipt-vlm-outputs.tgz outputs/preds outputs/metrics outputs/sft/best outputs/grpo/best

## 7. Release (Phase 8)

In [ ]:
!make merge
!uv run python scripts/push_to_hub.py --adapter outputs/sft/best --merged outputs/merged --repo <your-hf-user>/qwen3-vl-2b-receipts